# Lesson 1 — Linear regression, Karpathy-style

Companion to [`01_how_models_learn.py`](../01_how_models_learn.py) and [`aside/model_A_linear.py`](../aside/model_A_linear.py).

This notebook is a hands-on, step-by-step build of the simplest possible neural network: **two parameters** that learn to fit a line. No magic. Every tensor printed, every gradient hand-computed at step 0, every line explained.

By the end you will:

- Have **trained** a model with two parameters from scratch.
- Have computed its **loss and gradients by hand** at step 0 — and matched PyTorch's autograd.
- Have **inspected every intermediate tensor** along the way.
- Have compared gradient descent's answer to the **closed-form analytical solution**, proving they converge.
- Have a mental template for every later lesson: same loop, more parameters.

> 🔑 The 5-line training loop you build at the end of this notebook is **identical** to the one PRAGMA uses on a billion-parameter Transformer. The model changes; the loop doesn't.

---

## 🧰 What you should already know

- Basic Python (variables, functions, loops).
- High school algebra (slope-intercept form, `y = mx + b`).
- That's it. No calculus required — we'll derive the only derivative we need by hand.


## Step 0 — Imports and a fixed seed

A fixed random seed means every time you run this notebook (today, tomorrow, next month) you'll get the **exact same numbers**. That's important for a teaching notebook — you want the prints to match the text.

In [ ]:
import torch

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False)

print("torch version:", torch.__version__)

## Step 1 — Look at the data first

Before we even think about a model, let's look at what we're trying to fit.

Our task: there's a **secret rule** in the world. We don't know what it is. We can only observe input/output pairs. Our job is to recover the rule from the observations.

The secret rule is `y = 2x + 1`. We get to see 5 input/output pairs:

In [ ]:
x      = torch.tensor([1., 2., 3., 4., 5.])
y_true = 2 * x + 1   # the secret rule. The model will NOT see this line.

print("Input x:        ", x)
print("True output y:  ", y_true)
print()
print("As a table:")
print(f"  {'x':>5s} | {'y_true':>6s}")
print(f"  {'-'*5} | {'-'*6}")
for xi, yi in zip(x.tolist(), y_true.tolist()):
    print(f"  {xi:5.1f} | {yi:6.1f}")
print()
print(f"x.shape    = {tuple(x.shape)}")
print(f"y_true.shape = {tuple(y_true.shape)}")

**By eye:** can you see the rule? Each `y` is 2 times the `x` plus 1.

That's what the model has to figure out without being told.

## Step 2 — The simplest possible model: two parameters

A model is a **rule with knobs**. We propose the simplest rule that could possibly fit a straight line:

$$y = w \cdot x + b$$

Two knobs: `w` (the slope) and `b` (the intercept). We don't know what their right values are. We'll start them at **zero** — totally wrong — and let the computer figure it out.

The `requires_grad=True` flag tells PyTorch: *"this is a knob I want to tune later. Watch every calculation that involves it."* That's how `loss.backward()` later knows what to compute gradients for.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

print(f"w = {w.item()}   (slope, starts at 0)")
print(f"b = {b.item()}   (intercept, starts at 0)")
print()
print("The model says: y_pred = w*x + b = 0*x + 0 = 0 for every x.")
print()

# Test that
y_pred = w * x + b
print(f"y_pred for our 5 inputs: {y_pred.tolist()}")
print(f"y_true was:              {y_true.tolist()}")
print()
print("As bad as it gets — the model predicts 0 for everything.")

## Step 3 — Measure how wrong we are (the loss)

We need to turn "how wrong is the prediction" into a single number. That number is the **loss**.

The standard choice for predicting a number (regression) is **Mean Squared Error (MSE)**:

$$\mathrm{loss} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

In English: for each example, compute (predicted − true), square it, average over all examples.

**Why squared?**
- Squaring makes all errors positive (a miss of −5 hurts the same as a miss of +5).
- Squaring punishes big mistakes more than small ones — a miss of 10 contributes 100 to the loss; a miss of 1 contributes only 1.

Let's compute the loss step by step at our starting point (`w=0, b=0`):

In [ ]:
# By hand:
errors        = y_pred - y_true
errors_squared = errors ** 2
loss_manual   = errors_squared.mean()

print(f"y_pred:           {y_pred.tolist()}")
print(f"y_true:           {y_true.tolist()}")
print(f"errors:           {errors.tolist()}")
print(f"squared errors:   {errors_squared.tolist()}")
print(f"mean (loss):      {loss_manual.item():.4f}")
print()
# Verify with PyTorch's built-in
loss_torch = torch.nn.functional.mse_loss(y_pred, y_true)
print(f"PyTorch's MSE:    {loss_torch.item():.4f}")
print()
assert torch.allclose(loss_manual, loss_torch), "manual MSE != torch MSE"
print("✓ Our by-hand calculation matches PyTorch.")

**Quick mental check:**
- The errors are `[-3, -5, -7, -9, -11]` — we predicted 0 but the truth was `[3, 5, 7, 9, 11]`.
- Squared, they become `[9, 25, 49, 81, 121]`.
- Sum: 9+25+49+81+121 = 285. Mean: 285/5 = **57.0**.

That number (57.0) is how wrong we are. We want it to go to zero.

## Step 4 — Compute the gradients by hand

A **gradient** tells us: *"if I increase this knob by 1, how much does the loss change?"*

If the gradient is **positive**, increasing the knob makes loss go UP. To make loss go DOWN, we should DECREASE the knob. (And vice versa.)

For our specific loss formula `loss = mean((w*x + b - y_true)²)`, calculus gives the gradients:

$$\frac{\partial L}{\partial w} = \mathrm{mean}\left(2 \cdot \mathrm{error} \cdot x\right)$$

$$\frac{\partial L}{\partial b} = \mathrm{mean}\left(2 \cdot \mathrm{error}\right)$$

Don't worry if you don't see where these come from — it's just the chain rule applied to the loss formula. The important thing is **you don't have to remember them**. PyTorch will compute them for any expression you write. We'll just verify by hand once.

In [ ]:
# By hand:
grad_w_manual = (2 * errors * x).mean()
grad_b_manual = (2 * errors).mean()

print(f"errors:                {errors.tolist()}")
print(f"2 * errors * x:        {(2 * errors * x).tolist()}")
print(f"grad_w (mean of that): {grad_w_manual.item()}")
print()
print(f"2 * errors:            {(2 * errors).tolist()}")
print(f"grad_b (mean of that): {grad_b_manual.item()}")
print()

# Now let PyTorch compute them
loss_torch.backward()    # This populates w.grad and b.grad
print(f"PyTorch w.grad:        {w.grad.item()}")
print(f"PyTorch b.grad:        {b.grad.item()}")
print()
assert torch.allclose(grad_w_manual, w.grad), "manual grad_w != torch's"
assert torch.allclose(grad_b_manual, b.grad), "manual grad_b != torch's"
print("✓ PyTorch's autograd matches our by-hand calculus.")
print()
print("Reading the gradients:")
print(f"  grad_w = -50  →  the loss DECREASES when w gets BIGGER. So increase w.")
print(f"  grad_b = -14  →  the loss DECREASES when b gets BIGGER. So increase b.")

**The update rule.** We want to walk in the direction OPPOSITE to the gradient (since gradient points "uphill" and we want to go downhill):

$$w_{\text{new}} = w_{\text{old}} - \eta \cdot \frac{\partial L}{\partial w}$$

where η (eta) is the **learning rate** — how big a step to take.

Pick a small learning rate (0.05) and take one step:

In [ ]:
lr = 0.05

# Update by hand
w_new = w.item() - lr * w.grad.item()
b_new = b.item() - lr * b.grad.item()

print(f"  w: {w.item():.2f}  →  {w_new:.2f}    (update = {-lr * w.grad.item():+.2f})")
print(f"  b: {b.item():.2f}  →  {b_new:.2f}    (update = {-lr * b.grad.item():+.2f})")
print()

# Apply the update
with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad
    # IMPORTANT: zero the gradients for the next step — otherwise PyTorch
    # ACCUMULATES them, which would be wrong.
    w.grad.zero_()
    b.grad.zero_()

print(f"After one step: w = {w.item():.3f}, b = {b.item():.3f}")
print()
print("Predict again with the new w and b:")
with torch.no_grad():
    y_pred_new = w * x + b
    loss_new = ((y_pred_new - y_true) ** 2).mean()

print(f"  y_pred:    {y_pred_new.tolist()}")
print(f"  y_true:    {y_true.tolist()}")
print(f"  loss:      {loss_new.item():.4f}    (was 57.0 — dropped!)")

**That's it. That's machine learning.**

- We made a guess.
- We measured how wrong we were.
- PyTorch told us which way to nudge each knob.
- We nudged them.
- The loss went down.

Everything else in this course is the same idea, just with bigger models. Now let's wrap this up in a loop and run it many times.

## Step 5 — Wrap it in a 5-line loop

Five lines do the actual machine learning:

```python
y_pred = w * x + b                       # 1. guess
loss   = ((y_pred - y_true) ** 2).mean()  # 2. measure wrongness
opt.zero_grad()                          # 3. clear last step's gradient notes
loss.backward()                          # 4. compute gradients for w and b
opt.step()                               # 5. nudge w and b (= w -= lr * w.grad)
```

PyTorch provides an **Optimizer** object (`opt`) that bundles steps 3 and 5 for us, so we don't have to write `w -= lr * w.grad` ourselves.

Let's reset and train from scratch:

In [ ]:
# Reset — fresh w and b at 0
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

opt = torch.optim.SGD([w, b], lr=0.05)

# Track w, b, loss every 100 steps so we can plot them
history = {"step": [], "w": [], "b": [], "loss": []}

for step in range(1001):
    y_pred = w * x + b                            # 1
    loss   = ((y_pred - y_true) ** 2).mean()       # 2
    opt.zero_grad()                               # 3
    loss.backward()                                # 4
    opt.step()                                     # 5

    if step % 100 == 0:
        history["step"].append(step)
        history["w"].append(w.item())
        history["b"].append(b.item())
        history["loss"].append(loss.item())

print(f"{'step':>5}  {'w':>7}  {'b':>7}  {'loss':>10}")
print("-" * 36)
for i in range(len(history["step"])):
    print(f"{history['step'][i]:>5}  {history['w'][i]:>7.3f}  {history['b'][i]:>7.3f}  {history['loss'][i]:>10.6f}")
print()
print(f"Final: w = {w.item():.3f}, b = {b.item():.3f}    (target: w=2, b=1)")

**Read the table:**

- **Step 0** — `w` already at 2.5 (the first nudge was −0.05 × −50 = +2.5). Loss has dropped from 57 to 1.9.
- **Step 100** — `w` settles near 2.0; `b` is climbing slowly toward 1.0. Loss is in the thousandths.
- **Step 1000** — `w ≈ 2.00`, `b ≈ 0.99`. Loss near zero. We've found the rule.

`w` converges quickly because the gradient w.r.t. `w` was huge (-50). `b` converges slowly because its gradient was smaller (-14). This kind of "different parameters learn at different speeds" is a real thing — modern optimizers like AdamW try to balance it automatically.

## Step 6 — Visualise the loss curve

A picture of the loss going down over time is the most basic ML chart. Here's an ASCII version (no matplotlib needed).

In [ ]:
def ascii_plot(values, height=15, width=60, title=""):
    if title:
        print(title)
    vmin, vmax = min(values), max(values)
    rng = vmax - vmin if vmax != vmin else 1.0
    n = len(values)
    grid = [[" "] * width for _ in range(height)]
    for i, v in enumerate(values):
        col = int(i * (width - 1) / max(n - 1, 1))
        row = height - 1 - int((v - vmin) / rng * (height - 1))
        grid[row][col] = "*"
    print(f"{vmax:.4f} | " + "".join(grid[0]))
    for row in grid[1:-1]:
        print("        | " + "".join(row))
    print(f"{vmin:.4f} | " + "".join(grid[-1]))
    print("        +" + "-" * width)
    print("          step 0" + " " * (width - 16) + f"step {history['step'][-1]}")

ascii_plot(history["loss"], title="Loss over training (logarithmic y-axis-ish):")

The loss starts high and crashes toward zero — exactly what we want.

If the curve **plateaued** or went up, something would be wrong: learning rate too big, or model can't fit the data.

## Step 7 — Sanity check: compare to the closed-form solution

Here's a fun fact: this specific problem (fitting a line through points) has a **closed-form analytical solution**. You don't need gradient descent at all — you can just compute the answer with a formula.

This is called **Ordinary Least Squares (OLS)**:

$$w^* = \frac{N \sum x_i y_i - \sum x_i \sum y_i}{N \sum x_i^2 - (\sum x_i)^2}$$

$$b^* = \bar{y} - w^* \bar{x}$$

Let's compute it and check that our gradient descent found the same answer.

In [ ]:
N = len(x)
sum_x = x.sum().item()
sum_y = y_true.sum().item()
sum_xy = (x * y_true).sum().item()
sum_xx = (x * x).sum().item()

w_ols = (N * sum_xy - sum_x * sum_y) / (N * sum_xx - sum_x ** 2)
b_ols = (sum_y / N) - w_ols * (sum_x / N)

print(f"Closed-form solution (OLS):")
print(f"  w* = {w_ols:.6f}")
print(f"  b* = {b_ols:.6f}")
print()
print(f"Gradient descent (after 1000 steps):")
print(f"  w = {w.item():.6f}")
print(f"  b = {b.item():.6f}")
print()
print(f"Difference: |w - w*| = {abs(w.item() - w_ols):.4f}")
print(f"           |b - b*| = {abs(b.item() - b_ols):.4f}")
print()
print("Gradient descent has essentially found the exact analytical answer.")
print("The remaining tiny difference is because b hasn't fully converged yet —")
print("run for more steps and they'd match to many more decimals.")

**Why this matters:**

For linear regression, we don't need gradient descent at all — the closed-form solution is exact and much faster. But for **literally any model more complicated than this** (neural networks, transformers, anything non-linear), there is no closed-form solution. Gradient descent is the only option.

What you've just done — gradient descent on a problem we could have solved with a formula — is **identical** to what's happening when training GPT-4. The loop, the gradients, the parameter updates — all the same. Just with hundreds of billions of parameters and many more steps.

## Step 8 — Predict on new inputs

Once a model is trained, you can use it to predict on data it has never seen.

In [ ]:
new_x = torch.tensor([6.5, 10.0, 100.0, -3.0])

with torch.no_grad():
    new_y = w * new_x + b

print("Model's predictions on new inputs:")
print(f"  {'x':>6s} | {'y_pred':>8s} | {'true (2x+1)':>12s} | error")
print("  " + "-" * 50)
for xi, yi in zip(new_x.tolist(), new_y.tolist()):
    true = 2 * xi + 1
    print(f"  {xi:6.1f} | {yi:8.3f} | {true:12.3f} | {yi - true:+.4f}")

**Notice:** the model **extrapolates** — it correctly predicts `y` for `x = 100` even though it only ever saw `x` between 1 and 5. That's because we picked a model class (a line) that matches the underlying rule.

If we had trained a neural network on the same data and asked it to predict for `x = 100`, it would have probably given garbage — neural networks don't extrapolate well outside the range they were trained on. **Choosing the right model class for your data matters.**

## Step 9 — Visualise the loss surface

Since our model has only 2 parameters, we can actually **draw** the loss surface as a heatmap. The loss is `loss(w, b)` — a 2D function of the two knobs. Let's evaluate it on a grid and print it.

In [ ]:
def loss_at(w_val, b_val):
    return ((w_val * x + b_val - y_true) ** 2).mean().item()

# Make a grid of (w, b) values
w_range = [i * 0.4 for i in range(-1, 9)]    # w from -0.4 to 3.2
b_range = [i * 0.4 for i in range(-1, 6)]    # b from -0.4 to 2.0

print("Loss surface (lower = better):")
header_label = "b\\w"   # backslash workaround for f-string limitation
print(f"  {header_label:>6s} | " + " ".join(f"{wv:>6.1f}" for wv in w_range))
print("  " + "-" * (8 + 7 * len(w_range)))
for bv in reversed(b_range):
    row = f"  {bv:>6.1f} | "
    for wv in w_range:
        l = loss_at(wv, bv)
        # Mark the minimum spot (closest to (2, 1)) with [..]
        marker = ""
        if abs(wv - 2) < 0.2 and abs(bv - 1) < 0.2:
            marker = "*"
        row += f"{l:>6.2f}{marker:<1s}"
    print(row)
print()
print("The minimum is near (w=2.0, b=1.0). Gradient descent found it.")

## Step 10 — Things to try

Karpathy-style progressive exercises. Each builds on what we've just done.

### 🟢 Easy

1. **Change the secret rule.** Replace `y_true = 2 * x + 1` with `y_true = 5 * x - 3`. Rerun the notebook. What `w` and `b` does the model converge to?

2. **Change the learning rate.** Try `lr=0.001` (very slow) and `lr=0.5` (very fast). What happens to the loss curve? You should see slow convergence for the small lr, and oscillation or even divergence for the big lr.

3. **Change the initialization.** Start `w = torch.tensor(100.0, requires_grad=True)`. Can the model still find the answer? How many steps does it take?

### 🟡 Medium

4. **Add a third parameter — fit a quadratic.** Change the model to `y_pred = w2 * x**2 + w1 * x + b`. Generate `y_true` from a quadratic. Verify the model can fit it. (You'll have to add `w2` to the optimiser's parameter list.)

5. **Inspect the gradients at multiple steps.** Modify the training loop to also save `w.grad.item()` and `b.grad.item()` in `history`. After training, print them. The gradients should be HUGE at step 0 and tiny by step 1000 — that's how you know the model is converging.

6. **Implement SGD by hand.** Replace `opt.step()` with `w.data -= 0.05 * w.grad` and similarly for `b`. Verify training still works. You've just reimplemented `torch.optim.SGD` in two lines.

### 🔴 Hard (forward-pointers to later lessons)

7. **Use the autograd from L1d.** Replace PyTorch's `loss.backward()` with the `Value`-based autograd from `01d_autograd_from_scratch.py`. You'll need to do scalar-by-scalar computation instead of tensor ops. The result should match exactly.

8. **Try multivariate linear regression.** Make `x` be `(N, 3)` — three features per example. Make `w` a `(3,)` vector. The prediction becomes `y_pred = x @ w + b`. Train and verify it works. This is essentially one layer of a neural network.

9. **Add a non-linearity.** Make `y_pred = relu(w * x + b)`. Now you have a single neuron with a ReLU activation — the building block of every modern neural network. Generate data from `y = max(0, 2x - 5)` and verify the model can fit it.

## Summary

You just:

- ✅ Defined a model with two parameters (`w`, `b`) and a forward pass (`y = w*x + b`).
- ✅ Computed the loss (MSE) by hand and verified it matches PyTorch.
- ✅ Derived and computed the gradients **by hand** at step 0 — and matched PyTorch's autograd.
- ✅ Wrote the **5-line training loop** that is identical for every model in this course.
- ✅ Watched the loss go from 57 to nearly 0 over 1000 steps.
- ✅ Verified your trained model matches the closed-form OLS solution.
- ✅ Visualised the loss surface and saw the minimum.

**The recipe never changes.** Lesson 2 will use the same loop to train an embedding. Lesson 3 will use it to train an attention layer. Lesson 4 will use it to train a full Transformer. All the same five lines — just with more parameters.

Move on to:
- [**Lesson 1c**](../01c_gradient_descent.md) — even deeper on gradient descent, with an interactive visualisation.
- [**Lesson 1d**](../01d_autograd_from_scratch.py) — build a tiny autograd engine yourself.
- [**Lesson 2**](../02_tokens_and_embeddings.py) — how text becomes numbers.
